# 流特征提取

In [ ]:
import os
from scapy.utils import rdpcap
from scapy.layers.inet import IP, TCP
import torch  # For tensor storage
from scapy.all import *
from typing import Dict, Tuple

## 前置处理
根据 summary.txt 获取有效的五元组: `(src_ip, src_port, dst_ip, dst_port)`
为了兼容双向流，可以保证 `src_ip < dst_ip`

In [ ]:
# 过滤pcap文件，只提取指定四元组的报文
def build_vaild_flow_ids(summary_file):
    vaild_flow_ids = []
    with open(summary_file, "r") as f:
        lines = f.readlines()
        # 转换为四元组
        for line in lines:
            flow = eval(line)
            src_ip, src_port, dst_ip, dst_port = flow
            flow_id = (src_ip, src_port, dst_ip, dst_port)
            reverse_flow_id = (dst_ip, dst_port, src_ip, src_port)
            vaild_flow_ids.append(min(flow_id, reverse_flow_id))
    return vaild_flow_ids


# print(vaild_ids)

## 流特征提取

使用 scapy 库对流的各维度特征提取，以便后续构建样本，目前单流纬度特征包括：
1. 包长序列        packet_length
2. 包负载长度序列   payload_length
3. 包标记位序列     flags
4. 上行/下行       direction
5. 包时间戳序列     timestamp
6. 握手包内容(Server hello)       handshark   
7. 待完善...

In [ ]:
import pandas as pd


# Helper function to extract flow identifier
def get_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    return (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)


# 获取双向流的ID
def get_bidirectional_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    # Create a flow identifier
    flow_id = (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)
    reverse_flow_id = (ip_layer.dst, tcp_layer.dport, ip_layer.src, tcp_layer.sport)
    # Return the lexicographically smaller tuple to ensure consistency
    return min(flow_id, reverse_flow_id)


# Helper function to extract packet length
def get_packet_length(packet):
    if is_uplink(packet, get_bidirectional_flow_id(packet)) == "uplink":
        return len(packet)
    return -len(packet)


# Helper function to extract packet timestamp
def get_packet_timestamp(packet):
    return float(packet.time)


# Helper function to extract TCP flags
def get_tcp_flags(packet):
    return hex(int(packet[TCP].flags))


# Helper function to extract Payload Len
def get_payload_length(packet):
    if TCP in packet:
        # Get the payload of the TCP layer
        payload = packet[TCP].payload
        # Return the length of the payload
        return len(payload)
    return 0  # Return 0 if no payload exists


# Helper function to determine if a packet is uplink or downlink
def is_uplink(packet, flow_id):
    """
    Determine if a packet is uplink (client to server) or downlink (server to client).

    Args:
        packet: A Scapy packet object.
        flow_id: A tuple (src_ip, src_port, dst_ip, dst_port) representing the flow.

    Returns:
        str: "uplink" if the packet is client to server, "downlink" if server to client.
    """
    src_ip, src_port, dst_ip, dst_port = flow_id

    # Check if the packet matches the uplink direction
    if packet[IP].src == src_ip and packet[TCP].sport == src_port:
        return "uplink"  # Client to server

    # Check if the packet matches the downlink direction
    if packet[IP].src == dst_ip and packet[TCP].sport == dst_port:
        return "downlink"  # Server to client

    return "unknown"  # If it doesn't match either direction


# 提取流的 握手包内容, 需要具体内容
def extract_handshake_payload(packet) -> Tuple[Dict, bool]:
    # Read packets from the pcap file
    tls_data = {}
    vaild_handshake = False
    if not packet.haslayer("SSL/TLS"):
        return tls_data, False
    tls = packet.getlayer("SSL/TLS")
    if "TLS Record" not in tls:
        # print(f"No TLS Record found in packet: {packet.summary()}")
        return {}, False
    tls_data["tls_vers"] = hex(tls["TLS Record"].version)  # 0x303: TLS 1.2
    tls_data["tls_len"] = hex(tls["TLS Record"].length)
    if "TLS Handshake" in tls:
        tls_data["tls_step"] = tls["TLS Handshake"].type
        if tls_data["tls_step"] == 2:  # Server Hello
            # tls server hello length
            tls_data["tls_shlen"] = tls["TLS Handshake"].length
            # tls cipher ECDHE_RSA_WITH_AES_128_CBC_SHA256 0xc027 -> 49191
            tls_data["tls_cip"] = tls["TLS Handshake"].cipher_suite
            # tls compression method 0x00 -> 0
            tls_data["tls_comp"] = tls["TLS Handshake"].compression_method
            # tls extensions length 0x8 -> 8
            tls_data["tls_extlen"] = "0x" + str(tls["TLS Handshake"].extensions_length)
            # tls extensions type 0x000b -> 11
            tls_data["tls_exttype"] = tls["TLS Extension"].type
            vaild_handshake = True
        elif tls_data["tls_step"] == 11:  # Certificate Message
            tls.show()
            # tls_data['tls_certificate'] = tls['TLS Handshake']
        elif tls_data["tls_step"] == 12:  # Server Key Exchange todo
            tls.show()
        elif tls_data["tls_step"] == 14:  # Server Hello Done todo
            tls.show()
    return tls_data, vaild_handshake


# 提取流信息的函数
def extract_flows(
    pcap_file: str, extract_features: list, vaild_flow_ids=None
) -> Dict[str, Dict]:
    """
    Extract packet length sequences for each TCP flow from a pcap file.

    Args:
        pcap_file (str): Path to the pcap file.
        extract_features (list): List of features to extract from packets.

    Returns:
        dict: A dictionary where keys are flow identifiers (e.g., tuple of IPs and ports)
              and values are lists of packet lengths.
    """
    if not os.path.exists(pcap_file):
        raise FileNotFoundError(f"PCAP file not found: {pcap_file}")

    # Read packets from the pcap file
    packets = rdpcap(pcap_file)

    # 序列特征
    flows = {}  # Dictionary to store flows and their packet lengths
    # 切分成流, 流纬度特征提取
    for packet in packets:
        # Check if the packet has IP and TCP layers
        if IP in packet and TCP in packet:
            # print(f"Processing packet in flow: {bi_flow_id}")
            # flow_id = get_flow_id(packet) # 单向流
            flow_id = get_bidirectional_flow_id(packet)  # 双向流

            # 过滤背景流
            if vaild_flow_ids != None and flow_id not in vaild_flow_ids:
                continue
            raw_flow_id = tuple(flow_id)
            flow_id = str(flow_id)
            if flow_id not in flows:
                flows[flow_id] = {}
            if "flow_start_time" in extract_features:
                if "flow_start_time" not in flows[flow_id]:
                    flows[flow_id]["flow_start_time"] = get_packet_timestamp(packet)
            if "packet_length" in extract_features:
                flows[flow_id].setdefault("packet_length", []).append(
                    get_packet_length(packet)
                )
            if "timestamp" in extract_features:
                flows[flow_id].setdefault("timestamp", []).append(
                    get_packet_timestamp(packet)
                )
            if "flags" in extract_features:
                flows[flow_id].setdefault("flags", []).append(get_tcp_flags(packet))
            if "handshake" in extract_features:
                handshake_payload, vaild = extract_handshake_payload(packet)
                if vaild:
                    flows[flow_id].setdefault("handshake", []).append(handshake_payload)
            if "payload_length" in extract_features:
                flows[flow_id].setdefault("payload_length", []).append(
                    get_payload_length(packet)
                )
            if "direction" in extract_features:
                flows[flow_id].setdefault("direction", []).append(
                    is_uplink(packet, raw_flow_id)
                )

    return flows

## 样本拼接 & 持久化

分别对早期识别模型以及精细识别模型进行样本拼接；并通过 npy 格式进行持久化

- `encode_intra_inter_features_and_save` : 精细识别模型样本逻辑
- `encode_tls_features_and_save`: 早期识别模型样本构建逻辑


In [ ]:
import json
import pandas as pd
import numpy as np
from typing import List, Dict


# 自定义 JSON 编码器，处理 numpy 数据类型
class NumpyEncoder(json.JSONEncoder):
    """自定义 JSONEncoder，将 numpy 类型转换为 Python 原生类型"""

    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


# 保存成 tensor 张量 todo
def sink_tensors_file(flows, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # TLS Layer Feature
    """
        "tls_vers":"0x303",
        "tls_len":"0x7a",
        "tls_step":2,
        "tls_shlen":118,
        "tls_cip":4865,
        "tls_comp":0,
        "tls_extlen":46,
        "tls_exttype":43
    """

    tls_records: List[Dict] = []

    for i, (flow_id, feature) in enumerate(flows.items()):
        # Convert packet info to a PyTorch tensor
        handshake_payload = feature.get("handshake", [])
        if len(handshake_payload) > 0:
            tls_records.append(handshake_payload[0])

    fields = [
        "tls_vers",
        "tls_len",
        "tls_step",
        "tls_shlen",
        "tls_cip",
        "tls_comp",
        "tls_extlen",
        "tls_exttype",
    ]
    df = pd.DataFrame(tls_records)
    df = df.reindex(columns=fields)  # 保证列顺序并只留需要字段
    df = df.fillna("")  # 用默认值填充
    # 转为 numpy 字符串矩阵或做后续编码
    X_str = df.to_numpy(dtype=object)
    print(X_str)


def sink_json_file(flows, output_dir, idx: str):
    """
    Save flows as JSON files to the specified directory.

    Args:
        flows (dict): A dictionary where keys are flow identifiers and values are lists of packet info dicts.
        output_dir (str): Directory to save the JSON files.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    json.dump(
        flows,
        open(os.path.join(output_dir, "feature.json"), "w"),
        cls=NumpyEncoder,
        indent=4,
        separators=(",", ":"),
    )

In [ ]:
# Encoding TLS features: one-hot for certain fields, others to int and save as matrix
from locale import normalize
import os
import numpy as np
import joblib
import pandas as pd
from typing import List
from sklearn.preprocessing import OneHotEncoder
from typing import Type


def pad_trunc_1d(arr, L: int, pad_value=0.0, dtype: Type[np.floating] = np.float32):

    a = np.asarray(arr, dtype=dtype)
    if a.size >= L:
        return a[:L]
    out = np.full((L,), pad_value, dtype=dtype)
    out[: a.size] = a
    return out


def stats_18(x: np.ndarray) -> np.ndarray:
    """
    返回 18 维统计特征：
    min,max,mean,mad,std,var,skew,kurt, p10..p90(9个), count
    空输入 -> 全 0
    """
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return np.zeros(18, dtype=np.float32)

    # 基本统计
    xmin = np.min(x)
    xmax = np.max(x)
    mean = np.mean(x)
    mad = np.mean(np.abs(x - mean))
    var = np.var(x)
    std = np.sqrt(var)

    # skew/kurt（不依赖 scipy）
    # 注意：std==0 时避免除 0
    if std > 1e-12:
        z = (x - mean) / std
        skew = np.mean(z**3)
        kurt = np.mean(z**4) - 3.0
    else:
        skew = 0.0
        kurt = 0.0

    # 分位数（p10..p90）
    percs = np.percentile(x, np.arange(10, 100, 10)).astype(np.float32)

    cnt = np.float32(x.size)

    out = np.concatenate(
        [
            np.array([xmin, xmax, mean, mad, std, var, skew, kurt], dtype=np.float32),
            percs,
            np.array([cnt], dtype=np.float32),
        ]
    )
    return out  # (18,)


# todo(tyf) 尝试多种建边方式
def build_edges_time_threshold(
    flows_list, dt=1.0, add_self_loops=True, undirected=False
):
    """
    flows_list: [(fid, flow_dict), ...] 已经是选好/排序后的前 M 条
    return: edge_index np.ndarray shape [2, E] int64
    """
    starts = np.array(
        [flow.get("flow_start_time", 0.0) for _, flow in flows_list], dtype=np.float32
    )
    M = len(flows_list)

    src, dst = [], []

    for i in range(M):
        for j in range(M):
            if i == j:
                continue
            if abs(starts[i] - starts[j]) < dt:
                src.append(i)
                dst.append(j)

    if add_self_loops:
        src.extend(range(M))
        dst.extend(range(M))

    edge_index = np.array([src, dst], dtype=np.int64)

    if undirected:
        # 无向化：加反向边
        edge_index = np.hstack([edge_index, edge_index[::-1]])

    return edge_index


def second_phase_tensor_collect(
    flows: dict,
    instance_id: str,
    L: int = 128,
    M: int = 32,
    website_id: int = 0,
):
    """
    精细指纹阶段，从 flows 中提取 intra_inter 特征
    参数：
        flows: dict[fid, dict[feature_name, feature_val]], extract_flows 返回的 flows 结构
        output_dir: str, 保存输出的目录（会创建）
        M: int, 流的数量
        L: int, 处理的包长长度
        website_id: int, 网站类别 ID，暂未使用
    """
    X = np.zeros(
        (M, L + L + 3 * 18), dtype=np.float32
    )  # pkt_len + time_diffs + 3 * 18 stats

    flows_list = list(flows.items())[: min(M, len(flows))]
    # todo(tyf) 随机选取 M 条流，或按某种顺序选取

    for row_idx, (_, flow) in enumerate(flows_list):

        # pkt_lengths seq feature
        # 归一化
        MAX_LEN = 1500.0  # 或你想用的上限
        pkt_lengths = pad_trunc_1d(flow.get("packet_length", []), L, pad_value=0)
        pkt_lengths = pkt_lengths / MAX_LEN

        # time_diffs seq feature
        timestamps = pad_trunc_1d(
            flow.get("timestamp", []), L, pad_value=0.0, dtype=np.float64
        )
        times_diffs = np.zeros(L, dtype=np.float64)
        if L > 1:
            times_diffs[1:] = np.diff(timestamps)

        times_diffs = times_diffs.astype(np.float32)

        # statistics features
        np_pkt_lengths = np.array(pkt_lengths)
        outbound = np_pkt_lengths[np_pkt_lengths > 0]
        inbound = -np_pkt_lengths[np_pkt_lengths < 0]
        all_stats = stats_18(pkt_lengths[pkt_lengths != 0])  # 可选：排除 padding 0
        in_stats = stats_18(inbound)
        out_stats = stats_18(outbound)

        flow_vec = np.zeros((2 * L + 54,), dtype=np.float32)

        flow_vec[:L] = pkt_lengths
        flow_vec[L : 2 * L] = times_diffs
        flow_vec[2 * L : 2 * L + 18] = all_stats
        flow_vec[2 * L + 18 : 2 * L + 36] = in_stats
        flow_vec[2 * L + 36 :] = out_stats

        X[row_idx] = flow_vec

    # edge_index 构建并保存
    edge_index = build_edges_time_threshold(
        flows_list, dt=1.0, add_self_loops=True, undirected=False
    )

    Y = np.array([website_id], dtype=np.int64)

    assert X.shape == (M, L + L + 3 * 18)
    assert edge_index.shape[0] == 2
    assert Y.shape == (1,)

    return X.tolist(), edge_index.tolist(), Y.tolist()


def encode_tls_features_and_save(flows, output_dir, instance_id: str):
    """
    从 flows 中提取 TLS 握手记录，对指定字段做 one-hot 编码，其余数值字段转成 int，
    最终得到一个 numpy 矩阵并保存，同时保存 OneHotEncoder 对象以便推理时复用。
    参数：
        flows: dict, extract_flows 返回的 flows 结构
        output_dir: str, 保存输出的目录（会创建）
    返回： (X, ohe, fields_order)
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    # 期望字段顺序（与 sink_tensors_file 保持一致）
    fields = [
        "tls_vers",
        "tls_len",
        "tls_step",
        "tls_shlen",
        "tls_cip",
        "tls_comp",
        "tls_extlen",
        "tls_exttype",
    ]

    # 收集每条 flow 的第一个 handshake 字典（若存在）
    records = []
    for fid, feat in flows.items():
        hs = feat.get("handshake", [])
        if len(hs) > 0 and isinstance(hs[0], dict):
            records.append(hs[0])

    df = pd.DataFrame(records)
    # 保证列顺序并补空
    df = df.reindex(columns=fields)
    df = df.fillna("")

    # 把 hex 字符串(如 '0x7a') 或字符串数字 转为 int，对于不能转换的填 0
    def hex_to_int_safe(x):
        if x is None or x == "":
            return 0
        if isinstance(x, (int, float)):
            try:
                return int(x)
            except Exception:
                return 0
        s = str(x)
        if s.startswith("0x") or s.startswith("0X"):
            try:
                return int(s, 16)
            except Exception:
                return 0
        try:
            return int(float(s))
        except Exception:
            return 0

    # 需要 one-hot 的列
    onehot_cols = ["tls_vers", "tls_step", "tls_cip", "tls_comp", "tls_exttype"]
    # 需要作为数值 int 的列
    int_cols = ["tls_len", "tls_shlen", "tls_extlen"]

    # 先把 int_cols 转换为 int 数值
    for c in int_cols:
        if c in df.columns:
            df[c] = df[c].apply(hex_to_int_safe)
        else:
            df[c] = 0

    # 对 one-hot 列，先转为字符串（保证分类一致），缺失用特殊字符串 '__MISSING__'
    for c in onehot_cols:
        if c not in df.columns:
            df[c] = "__MISSING__"
        else:
            df[c] = df[c].apply(
                lambda x: "__MISSING__" if x is None or x == "" else str(x)
            )

    # fit OneHotEncoder: 兼容不同 sklearn 版本的参数名称
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    ohe_arr = ohe.fit_transform(df[onehot_cols])  # shape (n_samples, n_ohe_features)

    # 构建 int array
    int_arr = df[int_cols].to_numpy(dtype=np.int32)  # shape (n_samples, len(int_cols))

    # 合并为最终矩阵（one-hot 在前，int 在后）
    X = np.concatenate([ohe_arr, int_arr], axis=1) if int_arr.size else ohe_arr
    np.set_printoptions(suppress=True, precision=4, threshold=100000)
    # 保存结果与 encoder
    np.save(os.path.join(output_dir, f"X_tls_{instance_id}.npy"), X)
    joblib.dump(ohe, os.path.join(output_dir, f"ohe_tls.joblib"))

    # 记录列顺序信息便于推理时恢复（onehot 输出顺序 + int_cols）
    try:
        ohe_feature_names = ohe.get_feature_names_out(onehot_cols).tolist()
    except Exception:
        try:
            ohe_feature_names = list(ohe.get_feature_names(onehot_cols))
        except Exception:
            ohe_feature_names = []
            if hasattr(ohe, "categories_"):
                for i, c in enumerate(onehot_cols):
                    cats = ohe.categories_[i]
                    ohe_feature_names += [f"{c}__{v}" for v in cats]
    fields_order = ohe_feature_names + int_cols

    joblib.dump(fields_order, os.path.join(output_dir, "fields_order.joblib"))
    print(f"TLS feature fields order: {fields_order}")
    print("Saved X_tls.npy shape=", X.shape)
    return X, ohe, fields_order

In [ ]:
import os
import numpy as np
import json


def flush_shard(X_buf, y_buf, edge_buf, out_dir, shard_idx):
    # 1) X / y
    X = np.stack(X_buf).astype(np.float32)  # [N, M, D]
    y = np.asarray(y_buf, dtype=np.int64)  # [N]

    # 2) edges + edge_ptr
    edge_ptr = [0]
    for e in edge_buf:
        e = np.asarray(e, dtype=np.int64)
        edge_ptr.append(edge_ptr[-1] + e.shape[1])
    edge_ptr = np.asarray(edge_ptr, dtype=np.int64)  # [N+1]

    if edge_ptr[-1] == 0:
        edges = np.zeros((2, 0), dtype=np.int64)
    else:
        edges = np.concatenate(edge_buf, axis=1).astype(np.int64)  # [2, total_E]

    # 3) save
    np.save(os.path.join(out_dir, f"X_{shard_idx:03d}.npy"), X)
    np.save(os.path.join(out_dir, f"y_{shard_idx:03d}.npy"), y)
    np.save(os.path.join(out_dir, f"edges_{shard_idx:03d}.npy"), edges)
    np.save(os.path.join(out_dir, f"edge_ptr_{shard_idx:03d}.npy"), edge_ptr)

    # 4) log
    print(
        f"Flushed shard {shard_idx:03d}: N={X.shape[0]}, M={X.shape[1]}, D={X.shape[2]}, total_E={edges.shape[1]}"
    )

## 全链路链路测试

执行前置处理、特征提取、特征计算、全链路测试

In [ ]:
import os
import numpy as np
from random import sample
from re import X

# 特征提取主流程
root_dir = "/home/tyf/Project/encrypt_traffic/train_raw_data"
remote_root_dir = "/home/tyf/fnnas/Study/Traffic-data/train_raw_data"
output_dir = "/home/tyf/Project/Tantic/raw_feature"

web_instance_counter = {}
failed_instances = []


def draw_instance_counts(web_instance_counter):
    """绘制每个网站实例数量的柱状图"""
    import matplotlib.pyplot as plt

    websites = list(web_instance_counter.keys())
    counts = list(web_instance_counter.values())
    plt.figure(figsize=(12, 6))
    plt.bar(websites, counts)
    plt.xlabel("Website")
    plt.ylabel("Number of Instances")
    plt.title("Number of Instances per Website")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("./results/website_instance_counts.png")


def sample_iter():
    idx = 1
    website_idx = 0
    for website_name in os.listdir(remote_root_dir):
        website_idx = website_idx + 1
        website_folder = os.path.join(remote_root_dir, website_name)
        if not os.path.isdir(website_folder):
            continue

        for instance_id in os.listdir(website_folder):
            data_dir = os.path.join(website_folder, instance_id)
            if not os.path.isdir(data_dir):
                continue

            web_instance_counter[website_name] = (
                web_instance_counter.get(website_name, 0) + 1
            )

            # Find pcap file and summary file
            pcap_file = None
            summary_file = None
            for filename in os.listdir(data_dir):
                if filename == "traffic.pcap":
                    pcap_file = os.path.join(data_dir, filename)
                elif filename == "summary.txt":
                    summary_file = os.path.join(data_dir, filename)

            if pcap_file is None or summary_file is None:
                failed_instances.append(data_dir)
                continue

            # Build valid flow IDs
            vaild_flow_ids = build_vaild_flow_ids(summary_file)
            # Extract packet lengths for each TCP flow
            flows = extract_flows(
                pcap_file,
                extract_features=[
                    "packet_length",
                    "payload_length",
                    "direction",
                    "timestamp",
                    "flow_start_time",
                    "flags",
                    # "handshake",
                ],
                vaild_flow_ids=vaild_flow_ids,
            )

            # Save flows, Only for test
            # sink_json_file(flows, output_dir, f"{website_idx}-{instance_id}")

            # sink_tensors_file(flows, output_dir)
            # encode_tls_features_and_save(flows, output_dir, f"{website_idx}-{instance_id}")

            # 使用归一化的包长特征提取
            X_i, edge_i, y_i = second_phase_tensor_collect(
                flows,
                instance_id=f"{website_idx}-{instance_id}",
                L=128,
                M=32,
                website_id=website_idx,
            )

            # check all zero
            arr = np.array(X_i)
            if np.all((arr == 0) | np.isnan(arr)):
                continue

            idx = idx + 1
            if idx % 100 == 0:
                print(f"Processed instance {idx} for website {website_name}")

            yield X_i, edge_i, y_i


def save_dataset_sharded(
    sample_iter, out_dir, shard_size=30000, meta=None, shuffle=True, seed=42
):
    os.makedirs(out_dir, exist_ok=True)
    X_buf, y_buf, edge_index_buf = [], [], []
    shard_idx = 0
    rng = np.random.default_rng(seed)
    for X_i, edge_i, y_i in sample_iter:
        X_buf.append(X_i)
        edge_index_buf.append(edge_i)
        y_buf.append(y_i)
        if len(X_buf) >= shard_size:
            if shuffle:
                print("Shuffling shard", shard_idx, "with", len(X_buf), "samples")
                perm = rng.permutation(len(X_buf))
                X_buf = [X_buf[i] for i in perm]
                edge_index_buf = [edge_index_buf[i] for i in perm]
                y_buf = [y_buf[i] for i in perm]
            flush_shard(X_buf, y_buf, edge_index_buf, out_dir, shard_idx)
            shard_idx += 1
            X_buf, y_buf, edge_index_buf = [], [], []

    if X_buf:
        if shuffle:
            perm = rng.permutation(len(X_buf))
            X_buf = [X_buf[i] for i in perm]
            edge_index_buf = [edge_index_buf[i] for i in perm]
            y_buf = [y_buf[i] for i in perm]
        flush_shard(X_buf, y_buf, edge_index_buf, out_dir, shard_idx)

    if meta is None:
        meta = {}
    with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


# 1) build data - 使用归一化版本
save_dataset_sharded(
    sample_iter=sample_iter(), out_dir=output_dir, shard_size=50000, meta=None
)

# 2) dataset statistics
draw_instance_counts(web_instance_counter)
print("Failed instances:", failed_instances)

In [4]:

from curses import meta
from hmac import new
import numpy as np
from torch import zero_
import json

np.set_printoptions(suppress=True, precision=6)
base_dir = "/home/tyf/Project/Tantic/raw_feature/stgc_sp_all_class_tls_2"
X = np.load(f"{base_dir}/X_000.npy")
y = np.load(f"{base_dir}/y_000.npy")
edges = np.load(f"{base_dir}/edges_000.npy")
ptr = np.load(f"{base_dir}/edge_ptr_000.npy")
T = np.load(f"{base_dir}/T_000.npy")
meta = json.load(open(f"{base_dir}/meta.json", "r"))
print("meta:", meta)
print("X shape:, X[0] shape", X.shape, X[0].shape)
print("demo X[0][0]: ", X[0][0])
print("y[0]:", y[0])
print(edges[:, ptr[0] : ptr[1]])

# 检测全 0 样本
zero_count = 0
for i in range(X.shape[0]):
    if np.all(X[i][0] == 0):
        zero_count += 1

print("Zero sample check done.", zero_count)


print("y shape:", y.shape)
print("demo y:", y[:20])
# 统计 y 标签的数量
for label in np.unique(y):
    count = np.sum(y == label)
    print(f"Label {label}: {count} samples")

print("edges shape:", edges.shape)
print("edge_ptr shape:", ptr.shape)

uniq = np.unique(y)
print("y unique:", uniq)


print("=================================== Tls ================================")
print(f"T shape:", T.shape)
print("demo T:", T[:1])

# check T 全 0 样本
zero_T_count = 0
for i in range(T.shape[0]):
    if np.all(T[i][1] == 0):
        zero_T_count += 1
print("Zero T sample count:", zero_T_count)

# import os

# # 写一段函数处理 edge_ptr_000.npy, X_000.npy, y_000.npy, edges_000.npy 文件, 过滤 样本 X[i][0] 全 0 的样本, 并重新生成新的文件
# file_dir = "/home/tyf/Project/Tantic/raw_feature"
# X = np.load(os.path.join(file_dir, "X_000.npy"))
# y = np.load(os.path.join(file_dir, "y_000.npy"))
# edges = np.load(os.path.join(file_dir, "edges_000.npy"))
# ptr = np.load(os.path.join(file_dir, "edge_ptr_000.npy"))

# new_X = []
# new_y = []
# new_edges = []
# new_ptr = [0]
# for i in range(X.shape[0]):
#     if not np.all(X[i][0] == 0):
#         new_X.append(X[i])
#         new_y.append(y[i])
#         start = ptr[i]
#         end = ptr[i + 1]
#         new_edges.append(edges[:, start:end])
#         new_ptr.append(new_ptr[-1] + (end - start))

# new_X = np.array(new_X)
# new_y = np.array(new_y)
# new_edges = np.concatenate(new_edges, axis=1)
# new_ptr = np.array(new_ptr)

# np.save(os.path.join(file_dir, "X_001.npy"), new_X)
# np.save(os.path.join(file_dir, "y_001.npy"), new_y)
# np.save(os.path.join(file_dir, "edges_001.npy"), new_edges)
# np.save(os.path.join(file_dir, "edge_ptr_001.npy"), new_ptr)

# import os

# os.getpid()


meta: {'created_by': 'STGCGraphTensorCollector', 'created_time': '2026-01-17T04:53:10', 'collector_sample_file_dir': '/home/tyf/fnnas/Study/Traffic-data/train_raw_data', 'collector_num_flows_padding': 32, 'collector_num_packet_padding': 20, 'collector_edge_build_method': 'spatio_temporal', 'collector_node_feature_dim': 26, 'collector_tls_node_padding': 8, 'collector_tls_threshold': 1.0, 'sinker_shard_size': 50000, 'sinker_num_shards': 1, 'sinker_total_samples': 10763, 'sinker_shuffle': True}
X shape:, X[0] shape (10763, 32, 26) (32, 26)
demo X[0][0]:  [ 0.049333 -0.044     0.036     1.056667 -0.04     -0.04     -0.969333
  0.036    -1.833333  0.036    -0.500667  0.036     0.098    -0.04
 -0.218667 -0.04      0.044     1.515333 -0.04     -0.04      1.515333
  6.713333  1.1484    3.806     0.500667  0.346   ]
y[0]: [5]
[[ 0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1  1  2  2  2  2  2
   2  2  2  2  3  3  3  3  3  3  3  3  3  4  4  4  4  4  4  4  4  4  5  5
   5  5  5  5  5  5  5 

In [3]:
tls_fields = [
    "tls_vers",
    "tls_len",
    # ClientHello 字段
    "ch_shlen",
    "ch_cip",
    "ch_comp",
    "ch_extlen",
    "ch_exttype",
    "has_client_hello",
    # SNI 字段
    "sni_len",
    "sni_hash",
    "sni_label_count",
    "has_sni",
    # ServerHello 字段
    "sh_shlen",
    "sh_cip",
    "sh_comp",
    "sh_extlen",
    "sh_exttype",
    "has_server_hello",
    # Certificate 字段
    "cert_chain_len",
    "cert_count",
    "cert_len",
    "has_certificate",
    # Server Key Exchange 字段
    "ske_len",
    "ske_curve_type",
    "has_server_key_exchange",
    # Server Hello Done 字段
    "shd_len",
    "has_server_hello_done",
]

print(len(tls_fields))

27
